# Flow: filtered vs. unfiltered visual proprioception

Compares an unfiltered visual-proprioception regressor against exponentially
averaged and Kalman-filtered versions of the *same* regressor, on the same
held-out demonstrations.

The base encoder defaults to the **random-projection CNN** (`vgg19_rademacher_128`),
which is training-free: its backbone is frozen and its projection is a seeded
random matrix, so nothing about the sensor processor is learned. The only thing
this flow trains is the 64/64 MLP regressor, which takes seconds on cached
latents. That makes this the cheapest flow in the repository to run end to end,
and a fair place to measure a filter, since there is no encoder training that
could overlap the evaluation demonstrations.

The two filtered runs declare `base_run` and therefore train nothing at all:
they reuse the base run's trained regressor and its cached latents. All three
compared runs share identical weights and differ only in how the predicted
positions are filtered, so the difference in error is attributable to the
filter alone.

Filter parameters are tuned inside this flow, on the base run's **training**
demonstrations, and never on the evaluation set. This matters: the starting
values committed in `vp_ptun_vgg19_128_ema.yaml` and its Kalman counterpart
were derived for the proprioception-tuned VGG19, whose error is smaller than
the random-projection encoder's, and the optimal time constant grows with the
error.

See `DESIGN-TemporalVisualProprioception.md`.

Steps:
1. Set up the external flow directory and copy in the exp/run families needed,
   including `sensorprocessing_random_projection_cnn`.
2. Import the demopack and derive the data lists.
3. Generate the base `visual_proprioception` exp/run and train its regressor.
4. Tune the EMA and Kalman parameters on that run's training data.
5. Generate the two filtered exp/runs and the comparison collection.
6. Verify each of the three, then run the comparison.


In [1]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import yaml
import numpy as np
import tqdm
import papermill
import visproprio_helper
from demonstration.demopack import import_demopack, group_chooser_sp_vp_standard
from visual_proprioception.visproprio_helper import predict_positions
from visual_proprioception.visproprio_filters import (
    estimate_noise, tune_ema, tune_kalman)

FIELDS = ["height", "distance", "heading",
          "wrist_angle", "wrist_rotation", "gripper"]


ImportError: attempted relative import with no known parent package

## Set up the external flow directory

Creates `<flows_path>/<flow_name>/{expruns,results}` and points `Config` at it.
`external_setup` does not know about `sensorprocessing_random_projection_cnn`,
so it is copied here explicitly.


In [ ]:
flow_name = "FilteredVsUnfiltered_01"

# Use exist-ok not to re-run previously successfully run models, if this
# flow is executed again.
creation_style = "exist-ok"

expruns_path, results_path = visproprio_helper.external_setup(
    flow_name, pathlib.Path(Config()["flows_path"]).expanduser()
)

Config().copy_experiment("sensorprocessing_random_projection_cnn")


def run_notebook(entry):
    """Execute one queued step with papermill, inside the flow directory."""
    notebook_path = pathlib.Path("..", entry["notebook"])
    output_filename = (
        f"{notebook_path.stem}_{entry['experiment']}_{entry['run']}"
        f"_output{notebook_path.suffix}")
    output_path = pathlib.Path(results_path, output_filename)
    params = {
        "experiment": entry["experiment"],
        "run": entry["run"],
        "creation_style": creation_style,
        "expruns_path": expruns_path.as_posix(),
        "results_path": results_path.as_posix(),
    }
    print(f"*** Running {entry['notebook']} : "
          f"{entry['experiment']}/{entry['run']}")
    papermill.execute_notebook(
        notebook_path,
        output_path.absolute(),
        cwd=notebook_path.parent,
        parameters=params,
        kernel_name="berrypicker",
    )


## Import the demopack

`import_demopack` copies the demonstrations into the flow's results directory,
renaming them by group. The regressor trains on `vp_training` and everything is
evaluated on `vp_testing`. The `sp_training` group is unused here, because the
random-projection encoder is training-free.


In [ ]:
demopack_name = "random-both-cameras-video"
demonstration_cam = "dev2"

demopack_path = (
    pathlib.Path(Config()["demopacks_path"]).expanduser() / demopack_name)
selection = import_demopack(demopack_path, group_chooser_sp_vp_standard)


def demo_data(group):
    return [[demopack_name, demo, demonstration_cam] for demo in selection[group]]


vp_training_data = demo_data("vp_training")
vp_eval_data = demo_data("vp_testing")

print(f"regressor training demonstrations: {len(vp_training_data)}")
print(f"evaluation demonstrations:         {len(vp_eval_data)}")


## Parameters

The base encoder. The default is training-free, so the flow only ever trains
the regressor. The commented alternatives cost progressively more: the 256-dim
random projections are still training-free, while the proprioception-tuned
runs would additionally require a
`sensorprocessing/Train_ProprioTuned_CNN.ipynb` step to be queued before the
regressor (see the `Flow_PtunVsRandProj_128` flow for that pattern).


In [ ]:
latent_size = 128
epochs_vp = 1000

# Training-free encoders: no sensor-processing training step is needed.
base_encoder = {
    "slug": "randproj_vgg19_128",
    "name": "randproj-vgg19-128",
    "sensor_processing": "RandomProjectionCNNSensorProcessing",
    "sp_experiment": "sensorprocessing_random_projection_cnn",
    "sp_run": "vgg19_rademacher_128",
}
# base_encoder = {
#     "slug": "randproj_resnet50_128", "name": "randproj-resnet50-128",
#     "sensor_processing": "RandomProjectionCNNSensorProcessing",
#     "sp_experiment": "sensorprocessing_random_projection_cnn",
#     "sp_run": "resnet50_rademacher_128"}

# Marker-based, also training-free, but its latent depends on the markers
# being visible in the frame.
# base_encoder = {
#     "slug": "aruco_128", "name": "aruco-128",
#     "sensor_processing": "Aruco",
#     "sp_experiment": "sensorprocessing_aruco", "sp_run": "aruco_128"}

base_run = f"vp_{base_encoder['slug']}"
ema_run = f"{base_run}_ema"
kalman_run = f"{base_run}_kalman"
compare_run_name = f"comp_filter_{base_encoder['slug']}"

sample_interval = 0.1


## Generate and train the base regressor

This is the only training step in the flow. It has to finish before the filters
can be tuned, because tuning needs the regressor's predictions.


In [ ]:
def write_exprun(experiment, run, values):
    path = pathlib.Path(Config().get_exprun_path(), experiment, run + ".yaml")
    path.parent.mkdir(exist_ok=True, parents=True)
    with open(path, "w") as f:
        yaml.dump(values, f)
    return path


write_exprun("visual_proprioception", base_run, {
    "input-to-notebook": [
        "visual_proprioception/Train_VisualProprioception.ipynb",
        "visual_proprioception/Verify_VisualProprioception.ipynb",
    ],
    "name": base_encoder["name"],
    "output_size": 6,
    "proprioception_training_task": "proprio_regressor_training",
    "proprioception_testing_task": "proprio_regressor_validation",
    "encoding_size": latent_size,
    "regressor_hidden_size_1": 64,
    "regressor_hidden_size_2": 64,
    "loss": "MSE",
    "epochs": epochs_vp,
    "sample_interval": sample_interval,
    "position_filter": "none",
    "training_data": vp_training_data,
    "validation_data": vp_eval_data,
    "sensor_processing": base_encoder["sensor_processing"],
    "sp_experiment": base_encoder["sp_experiment"],
    "sp_run": base_encoder["sp_run"],
})

run_notebook({
    "notebook": "visual_proprioception/Train_VisualProprioception.ipynb",
    "experiment": "visual_proprioception",
    "run": base_run,
})


## Tune the filters on the training demonstrations

Both filters are tuned against `training_data`, never against the `vp_testing`
demonstrations the comparison reports on. The Kalman noise terms are measured
from the same data: measurement noise from the regressor's residual, process
noise from the acceleration of the commanded trajectory.


In [ ]:
exp_base = Config().get_experiment("visual_proprioception", base_run)
exp_robot = Config().get_experiment(
    exp_base["robot_exp"], exp_base["robot_run"])

training = predict_positions(exp_base, exp_robot, "training_data")
predictions = training["predictions"]
targets = training["targets"]
lengths = training["lengths"]

process_noise, measurement_noise = estimate_noise(
    predictions, targets, lengths, sample_interval)
ema_tuned = tune_ema(predictions, targets, lengths, sample_interval)
kalman_tuned = tune_kalman(predictions, targets, lengths, sample_interval,
                           process_noise, measurement_noise)

unfiltered = np.sqrt(np.mean((predictions - targets) ** 2, axis=0))
print(f"Tuned on {len(predictions)} training frames "
      f"in {len(lengths)} demonstrations\n")
print(f"{'field':<16}{'train RMSE':>12}{'ema':>9}{'tau':>8}"
      f"{'kalman':>10}{'noise x':>10}")
for index, field in enumerate(FIELDS):
    print(f"{field:<16}{unfiltered[index]:12.4f}"
          f"{ema_tuned['rmse'][index]:9.4f}{ema_tuned['parameters'][index]:8.2f}"
          f"{kalman_tuned['rmse'][index]:10.4f}"
          f"{kalman_tuned['parameters'][index]:10.2f}")


## Generate the filtered exp/runs and the comparison

Each filtered run only names its base and its filter settings. Everything else
- the sensor processor, the regressor geometry, the trained weights, the cached
latents - is resolved from `base_run`, so these runs train nothing.


In [ ]:
def write_filtered_run(run, suffix, filter_values):
    values = {
        "input-to-notebook": [
            "visual_proprioception/Tune_PositionFilter.ipynb",
            "visual_proprioception/Verify_VisualProprioception.ipynb",
            "visual_proprioception/Compare_VisualProprioception.ipynb",
        ],
        "name": f"{base_encoder['name']}-{suffix}",
        "base_experiment": "visual_proprioception",
        "base_run": base_run,
        "sample_interval": sample_interval,
    }
    values.update(filter_values)
    return write_exprun("visual_proprioception", run, values)


write_filtered_run(ema_run, "ema", {
    "position_filter": "ema",
    "filter_tau": [round(float(value), 4)
                   for value in ema_tuned["parameters"]],
})

write_filtered_run(kalman_run, "kalman", {
    "position_filter": "kalman",
    "filter_process_noise": [round(float(value), 6)
                             for value in process_noise],
    "filter_measurement_noise": [
        round(float(value), 6)
        for value in measurement_noise * kalman_tuned["parameters"]],
})

write_exprun("visual_proprioception_collections", compare_run_name, {
    "input-to-notebook": [
        "visual_proprioception/Compare_VisualProprioception.ipynb"],
    "name": compare_run_name,
    "tocompare": [base_run, ema_run, kalman_run],
    "proprioception_training_task": "proprio_regressor_training",
    "proprioception_testing_task": "proprio_regressor_validation",
})

entries = [
    {"notebook": "visual_proprioception/Verify_VisualProprioception.ipynb",
     "experiment": "visual_proprioception", "run": run}
    for run in (base_run, ema_run, kalman_run)
]
entries.append({
    "notebook": "visual_proprioception/Compare_VisualProprioception.ipynb",
    "experiment": "visual_proprioception_collections",
    "run": compare_run_name,
})
entries


## Run the verification and the comparison

`visual_proprioception/Verify_VisualProprioception.ipynb` produces a per-run `proprio_error.pdf` with
the filtered and unfiltered traces overlaid, and
`visual_proprioception/Compare_VisualProprioception.ipynb` produces the three-way plots and
`msecomparison_values.txt`.


In [ ]:
for entry in tqdm.tqdm(entries):
    try:
        run_notebook(entry)
    except Exception as e:
        print(f"There was an exception {e}")


## Result

The per-field RMSE of the three runs on the held-out `vp_testing`
demonstrations. Note that `visual_proprioception/Compare_VisualProprioception.ipynb` *appends* to
`msecomparison_values.txt`, so re-running this flow adds another block rather
than replacing the first.


In [ ]:
compare_exp = Config().get_experiment(
    "visual_proprioception_collections", compare_run_name)
values_file = pathlib.Path(compare_exp["data_dir"], "msecomparison_values.txt")
print(values_file)
print()
print(values_file.read_text())
